# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [1]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Make sure you have the correct paths

home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/"
downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/"
# downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Andersen/"
saved = originals + "saved/"
temp_files = originals + "temp/"
complete_files = downloads + "Cats/Datasets/Andersen/"

# Day we're updating data
update_date = "06-05-2025"

# Date range
date_range = "04-14-2025--06-05-2025"

os.chdir(downloads)

## Read Metadata 

In [2]:
# Read metadata

os.chdir(saved)
metadata_normalized = pd.read_csv("metadata_normalized.tsv", delimiter="\t") # Collection dates

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

print(len(metadata)) # 7397 rows

metadata = metadata.merge(metadata_normalized, how="outer")
print(metadata.columns)

metadata["name_state"] = metadata["geo_loc_name"].apply(lambda x: x.split("/")[1] if len(x.split("/")) > 1 else x.split("/")[0])

# Find only >= last date using Release Date from metadata 
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= datetime(2025, 4, 14).strftime("%Y-%m-%d")]
# Find only <= update date using Release Date from metadata
metadata = metadata[metadata["ReleaseDate"] <= datetime(2025, 6, 5).strftime("%Y-%m-%d")]

print(len(metadata)) # 6053 rows between 1/1/2024 and 4/14/2025
display(metadata)

9580
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
2157


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,create_date,version,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc,name_state
16776,SRR33124594,WGS,246.43,145473401,PRJNA1102327,SAMN47941264,Viral,50832578,USDA-NVSL,2025,...,2025-04-14 15:21:54,1,25-006243-005,SRP503016,NaN,"MILK, BULK TANK",SRS24711993,False,NaN,USA
16777,SRR33124594,WGS,246.43,145473401,PRJNA1102327,SAMN47941264,Viral,50832578,USDA-NVSL,2025,...,2025-04-14 15:21:54,1,25-006243-005,SRP503016,NaN,",",SRS24711993,False,NaN,
16778,SRR33124595,WGS,240.38,139928333,PRJNA1102327,SAMN47941263,Viral,48295907,USDA-NVSL,2025,...,2025-04-14 15:20:51,1,25-006243-002,SRP503016,NaN,"MILK, BULK TANK",SRS24711992,False,NaN,USA
16779,SRR33124595,WGS,240.38,139928333,PRJNA1102327,SAMN47941263,Viral,48295907,USDA-NVSL,2025,...,2025-04-14 15:20:51,1,25-006243-002,SRP503016,NaN,",",SRS24711992,False,NaN,
16780,SRR33124596,WGS,262.12,61927215,PRJNA1102327,SAMN47941262,Viral,21760157,USDA-NVSL,2025,...,2025-04-14 15:20:49,1,25-006243-001,SRP503016,NaN,"MILK, BULK TANK",SRS24711991,False,NaN,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18928,SRR33764587,WGS,148.55,209851920,PRJNA1207547,SAMN48808596,Viral,77089614,USDA-NVSL,2025,...,2025-05-30 12:24:46,1,25-015209-001,SRP557452,NaN,lung,SRS25207040,False,NaN,USA
18929,SRR33764588,WGS,146.62,75060770,PRJNA1207547,SAMN48808595,Viral,23409535,USDA-NVSL,2025,...,2025-05-30 12:24:41,1,25-015467-001,SRP557452,NaN,OROPHARYNGEAL SWAB,SRS25207039,False,NaN,USA
18930,SRR33764589,WGS,148.16,159798811,PRJNA1207547,SAMN48808594,Viral,59432778,USDA-NVSL,2025,...,2025-05-30 12:24:45,1,25-015388-002,SRP557452,NaN,CLOACAL/TRACHEAL SWAB POOL,SRS25207038,False,NaN,USA
18931,SRR33764590,WGS,148.93,170062068,PRJNA1207547,SAMN48808593,Viral,61180963,USDA-NVSL,2025,...,2025-05-30 12:24:44,1,25-013929-001,SRP557452,NaN,CLOACAL/TRACHEAL SWAB POOL,SRS25207037,False,NaN,USA


In [3]:
# Get list of genotypes

# os.chdir(home)

# genotypes_df = pd.read_excel("genotype_key.xlsx")

# genotypes = list(genotypes_df["Genotype"])

# print(genotypes)

genotypes = ["B3.13", "D1.1", "B3.2", "B3.6", "B3.7", "B3.5", "A3"]

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use genoflu_results.tsv

## Get genotype, specific geolocation

In [4]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")

genoflu_results["Run"] = genoflu_results["sample"]

metadata = metadata.merge(genoflu_results, on="Run", how="inner")
print(metadata)
# metadata = metadata[~metadata["Genotype"].str.contains('Not assigned')] # Do not include non-assigned genotypes
metadata = metadata[metadata["Genotype"].isin(genotypes)]

# Get only the genotypes we want: B3.13 and D1.1

# b313_and_d11_only = genoflu_results[(genoflu_results["Genotype"] == "B3.13") | (genoflu_results["Genotype"] == "D1.1")]
# b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
# b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")

# metadata = metadata.merge(b313_and_d11_only, on=["Run", "Genotype"], how="inner")

print(len(metadata)) 

display(metadata)

              Run Assay Type  AvgSpotLen      Bases    BioProject  \
0     SRR33124594        WGS      246.43  145473401  PRJNA1102327   
1     SRR33124594        WGS      246.43  145473401  PRJNA1102327   
2     SRR33124595        WGS      240.38  139928333  PRJNA1102327   
3     SRR33124595        WGS      240.38  139928333  PRJNA1102327   
4     SRR33124596        WGS      262.12   61927215  PRJNA1102327   
...           ...        ...         ...        ...           ...   
2152  SRR33764587        WGS      148.55  209851920  PRJNA1207547   
2153  SRR33764588        WGS      146.62   75060770  PRJNA1207547   
2154  SRR33764589        WGS      148.16  159798811  PRJNA1207547   
2155  SRR33764590        WGS      148.93  170062068  PRJNA1207547   
2156  SRR33764593        WGS      147.58  126434183  PRJNA1219588   

         BioSample BioSampleModel     Bytes Center Name Collection_Date  ...  \
0     SAMN47941264          Viral  50832578   USDA-NVSL            2025  ...   
1     SAMN4

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,name_state,sample,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List
0,SRR33124594,WGS,246.43,145473401,PRJNA1102327,SAMN47941264,Viral,50832578,USDA-NVSL,2025,...,USA,SRR33124594,2025-05-09_10-46-09,SRR33124594.fa,B3.13,"NS:am1.1, NP:am8, NA:ea1, MP:ea1, PA:ea1, PB1:...","am1.1:22-010085-001:NS, am8:23-032005-001:NP, ...","99.17%, 98.66%, 98.86%, 98.78%, 98.70%, 99.43%...","7, 20, 16, 12, 28, 13, 28, 33",Ran on FASTA - No Coverage Report
1,SRR33124594,WGS,246.43,145473401,PRJNA1102327,SAMN47941264,Viral,50832578,USDA-NVSL,2025,...,,SRR33124594,2025-05-09_10-46-09,SRR33124594.fa,B3.13,"NS:am1.1, NP:am8, NA:ea1, MP:ea1, PA:ea1, PB1:...","am1.1:22-010085-001:NS, am8:23-032005-001:NP, ...","99.17%, 98.66%, 98.86%, 98.78%, 98.70%, 99.43%...","7, 20, 16, 12, 28, 13, 28, 33",Ran on FASTA - No Coverage Report
2,SRR33124595,WGS,240.38,139928333,PRJNA1102327,SAMN47941263,Viral,48295907,USDA-NVSL,2025,...,USA,SRR33124595,2025-05-09_10-52-41,SRR33124595.fa,B3.13,"HA:ea1, NP:am8, NS:am1.1, PB2:am2.2, PB1:am4, ...","ea1:22-003707-003:HA, am8:23-032005-001:NP, am...","98.36%, 98.66%, 99.17%, 98.55%, 99.38%, 98.86%...","28, 20, 7, 33, 14, 16, 12, 28",Ran on FASTA - No Coverage Report
3,SRR33124595,WGS,240.38,139928333,PRJNA1102327,SAMN47941263,Viral,48295907,USDA-NVSL,2025,...,,SRR33124595,2025-05-09_10-52-41,SRR33124595.fa,B3.13,"HA:ea1, NP:am8, NS:am1.1, PB2:am2.2, PB1:am4, ...","ea1:22-003707-003:HA, am8:23-032005-001:NP, am...","98.36%, 98.66%, 99.17%, 98.55%, 99.38%, 98.86%...","28, 20, 7, 33, 14, 16, 12, 28",Ran on FASTA - No Coverage Report
6,SRR33124597,WGS,264.06,73438935,PRJNA1102327,SAMN47941261,Viral,26040226,USDA-NVSL,2025,...,USA,SRR33124597,2025-05-09_10-52-41,SRR33124597.fa,B3.13,"NP:am8, PA:ea1, MP:ea1, PB1:am4, PB2:am2.2, NA...","am8:23-032005-001:NP, ea1:22-003707-003:PA, ea...","99.00%, 98.75%, 98.78%, 99.38%, 98.64%, 98.86%...","15, 27, 12, 14, 31, 16, 7, 29",Ran on FASTA - No Coverage Report
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2152,SRR33764587,WGS,148.55,209851920,PRJNA1207547,SAMN48808596,Viral,77089614,USDA-NVSL,2025,...,USA,SRR33764587,2025-05-31_08-53-48,SRR33764587.fa,D1.1,"NP:am13, HA:ea3, NS:ea3, NA:am4N1, PB2:am24, P...","am13:24-030039-001:NP, ea3:22-013001-001:HA, e...","99.73%, 99.41%, 98.69%, 99.42%, 99.74%, 99.06%...","4, 10, 11, 6, 6, 17, 2, 5",Ran on FASTA - No Coverage Report
2153,SRR33764588,WGS,146.62,75060770,PRJNA1207547,SAMN48808595,Viral,23409535,USDA-NVSL,2025,...,USA,SRR33764588,2025-05-31_08-53-48,SRR33764588.fa,D1.1,"HA:ea3, PB2:am24, PA:am4, PB1:ea3, MP:ea3, NP:...","ea3:22-013001-001:HA, am24:24-030039-001:PB2, ...","99.35%, 99.56%, 99.37%, 99.30%, 99.69%, 99.67%...","11, 10, 11, 16, 3, 5, 11, 9",Ran on FASTA - No Coverage Report
2154,SRR33764589,WGS,148.16,159798811,PRJNA1207547,SAMN48808594,Viral,59432778,USDA-NVSL,2025,...,USA,SRR33764589,2025-05-31_08-53-49,SRR33764589.fa,D1.1,"NS:ea3, PB2:am24, NA:am4N1, HA:ea3, MP:ea3, PA...","ea3:22-013001-001:NS, am24:24-030039-001:PB2, ...","98.93%, 99.65%, 99.33%, 99.30%, 99.69%, 99.72%...","9, 8, 7, 12, 3, 5, 4, 13",Ran on FASTA - No Coverage Report
2155,SRR33764590,WGS,148.93,170062068,PRJNA1207547,SAMN48808593,Viral,61180963,USDA-NVSL,2025,...,USA,SRR33764590,2025-05-31_08-53-48,SRR33764590.fa,D1.1,"PA:am4, HA:ea3, MP:ea3, NS:ea3, NA:am4N1, NP:a...","am4:24-030039-001:PA, ea3:22-013001-001:HA, ea...","99.85%, 99.53%, 99.90%, 99.05%, 99.04%, 99.87%...","1, 8, 1, 8, 10, 2, 3, 3",Ran on FASTA - No Coverage Report


In [5]:
# Get specific geolocation and name_state from genbank_mapping.tsv

genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["Run"] = genbank_mapping["sra_run"]
genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first") # Drop duplicates
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

# print(genbank_mapping)

# metadata_genbank = metadata.merge(genbank_mapping, on=["Run"]) # Only include data that has states

# Get geolocation for second state attribute

os.chdir(home + "references/")
state_ref = pd.read_csv("states_ref.csv")
metadata["Geo_Location"] = metadata["name_state"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"] == x, 'Country'].iloc[0] + "-" + x if x in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x, 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x, 'Abbreviation'].iloc[0] if x in state_ref["State"].values else x)


print(metadata["name_state"])
print(len(metadata))
display(metadata) # Maybe there is no state information since 3/18/2025?

0       USA
1          
2       USA
3          
6       USA
       ... 
2152    USA
2153    USA
2154    USA
2155    USA
2156    USA
Name: name_state, Length: 1864, dtype: object
1864


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,sample,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,Geo_Location
0,SRR33124594,WGS,246.43,145473401,PRJNA1102327,SAMN47941264,Viral,50832578,USDA-NVSL,2025,...,SRR33124594,2025-05-09_10-46-09,SRR33124594.fa,B3.13,"NS:am1.1, NP:am8, NA:ea1, MP:ea1, PA:ea1, PB1:...","am1.1:22-010085-001:NS, am8:23-032005-001:NP, ...","99.17%, 98.66%, 98.86%, 98.78%, 98.70%, 99.43%...","7, 20, 16, 12, 28, 13, 28, 33",Ran on FASTA - No Coverage Report,USA
1,SRR33124594,WGS,246.43,145473401,PRJNA1102327,SAMN47941264,Viral,50832578,USDA-NVSL,2025,...,SRR33124594,2025-05-09_10-46-09,SRR33124594.fa,B3.13,"NS:am1.1, NP:am8, NA:ea1, MP:ea1, PA:ea1, PB1:...","am1.1:22-010085-001:NS, am8:23-032005-001:NP, ...","99.17%, 98.66%, 98.86%, 98.78%, 98.70%, 99.43%...","7, 20, 16, 12, 28, 13, 28, 33",Ran on FASTA - No Coverage Report,
2,SRR33124595,WGS,240.38,139928333,PRJNA1102327,SAMN47941263,Viral,48295907,USDA-NVSL,2025,...,SRR33124595,2025-05-09_10-52-41,SRR33124595.fa,B3.13,"HA:ea1, NP:am8, NS:am1.1, PB2:am2.2, PB1:am4, ...","ea1:22-003707-003:HA, am8:23-032005-001:NP, am...","98.36%, 98.66%, 99.17%, 98.55%, 99.38%, 98.86%...","28, 20, 7, 33, 14, 16, 12, 28",Ran on FASTA - No Coverage Report,USA
3,SRR33124595,WGS,240.38,139928333,PRJNA1102327,SAMN47941263,Viral,48295907,USDA-NVSL,2025,...,SRR33124595,2025-05-09_10-52-41,SRR33124595.fa,B3.13,"HA:ea1, NP:am8, NS:am1.1, PB2:am2.2, PB1:am4, ...","ea1:22-003707-003:HA, am8:23-032005-001:NP, am...","98.36%, 98.66%, 99.17%, 98.55%, 99.38%, 98.86%...","28, 20, 7, 33, 14, 16, 12, 28",Ran on FASTA - No Coverage Report,
6,SRR33124597,WGS,264.06,73438935,PRJNA1102327,SAMN47941261,Viral,26040226,USDA-NVSL,2025,...,SRR33124597,2025-05-09_10-52-41,SRR33124597.fa,B3.13,"NP:am8, PA:ea1, MP:ea1, PB1:am4, PB2:am2.2, NA...","am8:23-032005-001:NP, ea1:22-003707-003:PA, ea...","99.00%, 98.75%, 98.78%, 99.38%, 98.64%, 98.86%...","15, 27, 12, 14, 31, 16, 7, 29",Ran on FASTA - No Coverage Report,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2152,SRR33764587,WGS,148.55,209851920,PRJNA1207547,SAMN48808596,Viral,77089614,USDA-NVSL,2025,...,SRR33764587,2025-05-31_08-53-48,SRR33764587.fa,D1.1,"NP:am13, HA:ea3, NS:ea3, NA:am4N1, PB2:am24, P...","am13:24-030039-001:NP, ea3:22-013001-001:HA, e...","99.73%, 99.41%, 98.69%, 99.42%, 99.74%, 99.06%...","4, 10, 11, 6, 6, 17, 2, 5",Ran on FASTA - No Coverage Report,USA
2153,SRR33764588,WGS,146.62,75060770,PRJNA1207547,SAMN48808595,Viral,23409535,USDA-NVSL,2025,...,SRR33764588,2025-05-31_08-53-48,SRR33764588.fa,D1.1,"HA:ea3, PB2:am24, PA:am4, PB1:ea3, MP:ea3, NP:...","ea3:22-013001-001:HA, am24:24-030039-001:PB2, ...","99.35%, 99.56%, 99.37%, 99.30%, 99.69%, 99.67%...","11, 10, 11, 16, 3, 5, 11, 9",Ran on FASTA - No Coverage Report,USA
2154,SRR33764589,WGS,148.16,159798811,PRJNA1207547,SAMN48808594,Viral,59432778,USDA-NVSL,2025,...,SRR33764589,2025-05-31_08-53-49,SRR33764589.fa,D1.1,"NS:ea3, PB2:am24, NA:am4N1, HA:ea3, MP:ea3, PA...","ea3:22-013001-001:NS, am24:24-030039-001:PB2, ...","98.93%, 99.65%, 99.33%, 99.30%, 99.69%, 99.72%...","9, 8, 7, 12, 3, 5, 4, 13",Ran on FASTA - No Coverage Report,USA
2155,SRR33764590,WGS,148.93,170062068,PRJNA1207547,SAMN48808593,Viral,61180963,USDA-NVSL,2025,...,SRR33764590,2025-05-31_08-53-48,SRR33764590.fa,D1.1,"PA:am4, HA:ea3, MP:ea3, NS:ea3, NA:am4N1, NP:a...","am4:24-030039-001:PA, ea3:22-013001-001:HA, ea...","99.85%, 99.53%, 99.90%, 99.05%, 99.04%, 99.87%...","1, 8, 1, 8, 10, 2, 3, 3",Ran on FASTA - No Coverage Report,USA


In [6]:
# # If no states

# metadata_genbank = metadata

# metadata_genbank["name_state"] = "USA"

# metadata_genbank["Geo_Location"] = "USA"

# display(metadata_genbank)

## Collection Dates

If date is N/A, try finding it first

In [ ]:

# Get all dates
# metadata["Collection_Date_Specific"] = metadata["BioSample"].apply(lambda x: search_collection_date(x, metadata) if "-" not in x else x)

# Save this so we don't have to do it again

# os.chdir(temp_files)
# metadata.to_csv("metadata_genbank.csv")

https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_6841fc9d1daf9fb804024f89&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
2025-02-18
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_6841fca0b2a871884a0e2f13&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
2025-02-18
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_6841fca3859ccb22a801abb3&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
2025-02-18
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_6841fca58df6e0882c08d951&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
2025-02-18
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_6841fcad5a8ada2580075201&

In [11]:
# # Upload saved data -- if doing this, make sure the above cell is commented out
# os.chdir(temp_files + "saved/")
# metadata_genbank = pd.read_csv("metadata_genbank.csv")
# os.chdir(temp_files)

# # Get only updated dates

# unknown_dates = metadata_genbank[(metadata_genbank["Collection_Date_Specific"] == "2024") | (metadata_genbank["Collection_Date_Specific"] == "2025")] # Dates we don't have
# known_dates = metadata_genbank[(metadata_genbank["Collection_Date_Specific"] != "2024") & (metadata_genbank["Collection_Date_Specific"] != "2025")] # Dates we've already gotten

# # Get new dates also 
# # new_dates = metadata_genbank["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank) if )

# updated_unknown_dates = unknown_dates["BioSample"].apply(lambda x: search_collection_date(x, unknown_dates)) # Update unknown dates, if possible

# metadata_genbank = pd.concat([known_dates, unknown_dates], ignore_index=True, sort=True)

# # Remove pre-2024 dates

# metadata_genbank["Collection_Date_Compare"] = metadata_genbank["Collection_Date_Specific"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# metadata_genbank = metadata_genbank[metadata_genbank["Collection_Date_Compare"] >= datetime(2024, 1, 1).strftime("%Y-%m-%d")]

print(metadata[["Collection_Date_Specific"]])

display(metadata)

     Collection_Date_Specific
0                  2025-02-18
1                  2025-02-18
2                  2025-02-18
3                  2025-02-18
6                        2025
...                       ...
2152                     2025
2153                     2025
2154                     2025
2155                     2025
2156                     2025

[1864 rows x 1 columns]


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,Geo_Location,Collection_Date_Specific
0,SRR33124594,WGS,246.43,145473401,PRJNA1102327,SAMN47941264,Viral,50832578,USDA-NVSL,2025,...,2025-05-09_10-46-09,SRR33124594.fa,B3.13,"NS:am1.1, NP:am8, NA:ea1, MP:ea1, PA:ea1, PB1:...","am1.1:22-010085-001:NS, am8:23-032005-001:NP, ...","99.17%, 98.66%, 98.86%, 98.78%, 98.70%, 99.43%...","7, 20, 16, 12, 28, 13, 28, 33",Ran on FASTA - No Coverage Report,USA,2025-02-18
1,SRR33124594,WGS,246.43,145473401,PRJNA1102327,SAMN47941264,Viral,50832578,USDA-NVSL,2025,...,2025-05-09_10-46-09,SRR33124594.fa,B3.13,"NS:am1.1, NP:am8, NA:ea1, MP:ea1, PA:ea1, PB1:...","am1.1:22-010085-001:NS, am8:23-032005-001:NP, ...","99.17%, 98.66%, 98.86%, 98.78%, 98.70%, 99.43%...","7, 20, 16, 12, 28, 13, 28, 33",Ran on FASTA - No Coverage Report,,2025-02-18
2,SRR33124595,WGS,240.38,139928333,PRJNA1102327,SAMN47941263,Viral,48295907,USDA-NVSL,2025,...,2025-05-09_10-52-41,SRR33124595.fa,B3.13,"HA:ea1, NP:am8, NS:am1.1, PB2:am2.2, PB1:am4, ...","ea1:22-003707-003:HA, am8:23-032005-001:NP, am...","98.36%, 98.66%, 99.17%, 98.55%, 99.38%, 98.86%...","28, 20, 7, 33, 14, 16, 12, 28",Ran on FASTA - No Coverage Report,USA,2025-02-18
3,SRR33124595,WGS,240.38,139928333,PRJNA1102327,SAMN47941263,Viral,48295907,USDA-NVSL,2025,...,2025-05-09_10-52-41,SRR33124595.fa,B3.13,"HA:ea1, NP:am8, NS:am1.1, PB2:am2.2, PB1:am4, ...","ea1:22-003707-003:HA, am8:23-032005-001:NP, am...","98.36%, 98.66%, 99.17%, 98.55%, 99.38%, 98.86%...","28, 20, 7, 33, 14, 16, 12, 28",Ran on FASTA - No Coverage Report,,2025-02-18
6,SRR33124597,WGS,264.06,73438935,PRJNA1102327,SAMN47941261,Viral,26040226,USDA-NVSL,2025,...,2025-05-09_10-52-41,SRR33124597.fa,B3.13,"NP:am8, PA:ea1, MP:ea1, PB1:am4, PB2:am2.2, NA...","am8:23-032005-001:NP, ea1:22-003707-003:PA, ea...","99.00%, 98.75%, 98.78%, 99.38%, 98.64%, 98.86%...","15, 27, 12, 14, 31, 16, 7, 29",Ran on FASTA - No Coverage Report,USA,2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2152,SRR33764587,WGS,148.55,209851920,PRJNA1207547,SAMN48808596,Viral,77089614,USDA-NVSL,2025,...,2025-05-31_08-53-48,SRR33764587.fa,D1.1,"NP:am13, HA:ea3, NS:ea3, NA:am4N1, PB2:am24, P...","am13:24-030039-001:NP, ea3:22-013001-001:HA, e...","99.73%, 99.41%, 98.69%, 99.42%, 99.74%, 99.06%...","4, 10, 11, 6, 6, 17, 2, 5",Ran on FASTA - No Coverage Report,USA,2025
2153,SRR33764588,WGS,146.62,75060770,PRJNA1207547,SAMN48808595,Viral,23409535,USDA-NVSL,2025,...,2025-05-31_08-53-48,SRR33764588.fa,D1.1,"HA:ea3, PB2:am24, PA:am4, PB1:ea3, MP:ea3, NP:...","ea3:22-013001-001:HA, am24:24-030039-001:PB2, ...","99.35%, 99.56%, 99.37%, 99.30%, 99.69%, 99.67%...","11, 10, 11, 16, 3, 5, 11, 9",Ran on FASTA - No Coverage Report,USA,2025
2154,SRR33764589,WGS,148.16,159798811,PRJNA1207547,SAMN48808594,Viral,59432778,USDA-NVSL,2025,...,2025-05-31_08-53-49,SRR33764589.fa,D1.1,"NS:ea3, PB2:am24, NA:am4N1, HA:ea3, MP:ea3, PA...","ea3:22-013001-001:NS, am24:24-030039-001:PB2, ...","98.93%, 99.65%, 99.33%, 99.30%, 99.69%, 99.72%...","9, 8, 7, 12, 3, 5, 4, 13",Ran on FASTA - No Coverage Report,USA,2025
2155,SRR33764590,WGS,148.93,170062068,PRJNA1207547,SAMN48808593,Viral,61180963,USDA-NVSL,2025,...,2025-05-31_08-53-48,SRR33764590.fa,D1.1,"PA:am4, HA:ea3, MP:ea3, NS:ea3, NA:am4N1, NP:a...","am4:24-030039-001:PA, ea3:22-013001-001:HA, ea...","99.85%, 99.53%, 99.90%, 99.05%, 99.04%, 99.87%...","1, 8, 1, 8, 10, 2, 3, 3",Ran on FASTA - No Coverage Report,USA,2025


In [12]:
# If no collection dates

# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["Collection_Date"]

## Get host type

In [14]:
# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(downloads)

animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['owl', 'western gull', 'great horned owl', 'falcon', 'vulture', 'american black duck', 'flamingo', "'", 'tundra swan', 'red-shouldered hawk', 'cattle', 'blue jay', 'bufflehead', 'red-tailed hawk', 'goose', 'wood duck', 'cackling goose', 'gull', 'red-breasted merganser', 'turkey vulture', 'partridge', 'sharp-shinned hawk', '-', 'gadwall', 'cougar', 'duck', 'snow goose', 'american crow', 'bobcat', 'sanderling', 'trumpeter swan', 'western sandpiper', 'dunlin', 'barn owl', 'lesser scaup', 'cat', 'great blue heron', "bonaparte's gull", 'peregrine', 'bald eagle', 'mallard x american black duck hybrid', 'swan', 'american coot', 'hooded merganser', 'quail', 'greater scaup', 'common raven', "cooper's hawk", 'fox', 'snowy egret', 'mallard', 'barred owl', 'pet food', 'glaucous gull', 'black vulture', 'snowy owl', 'great black-backed gull', 'common loon', 'rock goose', 'skunk', 'canada goose', 'mute swan', 'sand crane', nan, 'mink', "ross's goose", 'great egret', 'crow', 'guinea', 'hawk', 'raccoo

In [15]:
# Get animals from animal reference
os.chdir(downloads)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata, animals_ref) # Get host type

metadata["years"] = metadata["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

## Make names using all the attributes we collected

In [16]:
for num, collection_date in enumerate(metadata["Collection_Date"]):
    
    if collection_date != collection_date: # If nan
        metadata.loc[num, "Collection_Date"] = metadata.loc[num, "years"]
    else: # If actual date
        if len(str(collection_date)) == 4: # If it's a year
            # print("caught")
            metadata.loc[num, "Collection_Date"] = collection_date
        else:
            parsed_date = dateutil.parser.parse(collection_date)
            date = parsed_date.strftime("%Y-%m-%d") # Make sure it doesn't default to today, if just a year
            metadata.loc[num, "Collection_Date"] = date

    metadata = metadata.dropna(thresh=2)

# Make names

# + metadata["BioSample"] + "|" 
names = ">" + metadata["Run"] + "|" + "A/" + metadata["Host"] + "/" + metadata["name_state"] + "/" + metadata["isolate"] + "/" + metadata["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata["Geo_Location"] + "|" + metadata["Collection_Date"].apply(lambda x: str(x)) + "|" + metadata["Host_Type"] + "|" + metadata["Genotype"]

metadata["Name"] = names

# metadata_genbank.to_csv("metadata_genbank_named.csv")

display(metadata[["ReleaseDate", 'create_date', 'Collection_Date']])

,ReleaseDate,create_date,Collection_Date
0,2025-04-14,2025-04-14 15:21:54,2025
1,2025-04-14,2025-04-14 15:21:54,2025
2,2025-04-14,2025-04-14 15:20:51,2025
3,2025-04-14,2025-04-14 15:20:51,2025
6,2025-04-14,2025-04-14 15:21:19,2025
...,...,...,...
2152,2025-05-30,2025-05-30 12:24:46,2025
2153,2025-05-30,2025-05-30 12:24:41,2025
2154,2025-05-30,2025-05-30 12:24:45,2025
2155,2025-05-30,2025-05-30 12:24:44,2025


In [18]:
# Drop duplicate runs 
metadata = metadata.drop_duplicates(subset="Run", keep="first")

In [19]:
print(metadata)

              Run Assay Type  AvgSpotLen        Bases    BioProject  \
0     SRR33124594        WGS      246.43  145473401.0  PRJNA1102327   
2     SRR33124595        WGS      240.38  139928333.0  PRJNA1102327   
6     SRR33124597        WGS      264.06   73438935.0  PRJNA1102327   
8     SRR33124598        WGS      255.94   76531946.0  PRJNA1102327   
10    SRR33124599        WGS      145.79   94820034.0  PRJNA1102327   
...           ...        ...         ...          ...           ...   
2152  SRR33764587        WGS      148.55  209851920.0  PRJNA1207547   
2153  SRR33764588        WGS      146.62   75060770.0  PRJNA1207547   
2154  SRR33764589        WGS      148.16  159798811.0  PRJNA1207547   
2155  SRR33764590        WGS      148.93  170062068.0  PRJNA1207547   
2156  SRR33764593        WGS      147.58  126434183.0  PRJNA1219588   

         BioSample BioSampleModel       Bytes Center Name Collection_Date  \
0     SAMN47941264          Viral  50832578.0   USDA-NVSL            2

## Make FASTA files

In [20]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata[metadata["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata[metadata["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

In [21]:
# print(fasta_files.keys())

In [22]:
# Create fasta files 

# os.chdir(complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/")
os.chdir(complete_files)
names = []
for pair in fasta_files.keys():
    # output_path = complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/" + pair + "_andersen_updated_" + update_date + ".fasta" 
    output_path = complete_files + pair + "_andersen_updated_" + update_date + ".fasta"

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        names.append(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

print(len(names)/8)

>SRR33124594|A/CATTLE/USA/25-006243-005/2025|H5N1|USA|2025|cattle|B3.13
>SRR33124595|A/CATTLE/USA/25-006243-002/2025|H5N1|USA|2025|cattle|B3.13
>SRR33124597|A/CATTLE/USA/25-006240-005/2025|H5N1|USA|2025|cattle|B3.13
>SRR33124598|A/CATTLE/USA/25-006031-002/2025|H5N1|USA|2025|cattle|B3.13
>SRR33124599|A/CAT/USA/25-007097-002/2025|H5N1|USA|2025|feline|B3.13
>SRR33124600|A/CAT/USA/25-007097-001/2025|H5N1|USA|2025|feline|B3.13
>SRR33124601|A/CAT/USA/25-006638-001/2025|H5N1|USA|2025|feline|B3.13
>SRR33124602|A/CAT/USA/25-006544-001/2025|H5N1|USA|2025|feline|B3.13
>SRR33124603|A/CAT/USA/25-006543-001/2025|H5N1|USA|2025|feline|B3.13
>SRR33124604|A/CATTLE/USA/25-006954-002/2025|H5N1|USA|2024-11-19|cattle|B3.13
nan
>SRR33124606|A/CATTLE/USA/25-006029-002/2025|H5N1|USA|2024-11-19|cattle|B3.13
nan
nan
nan
>SRR33124610|A/CATTLE/USA/25-006276-004/2025|H5N1|USA|2025|cattle|B3.13
>SRR33124611|A/CATTLE/USA/25-006276-003/2025|H5N1|USA|2025|cattle|B3.13
>SRR33124612|A/CATTLE/USA/25-006276-002/2025|H5N1|U

## De-Duplication

In [40]:
# De-duplication 

# Gisaid 

# gisaid = downloads + "GISAID/complete/B3_13_D1_1/" + date_range + "_B3_13_D1_1_North_America/"

gisaid = downloads + "Cats/Datasets/GISAID/"

os.chdir(gisaid)

dfs_gisaid = create_dataframes(gisaid)
# dfs_gisaid2 = create_dataframes(gisaid2)

A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.6_HA
B3.6_MP
B3.6_NA
B3.6_NP
B3.6_NS
B3.6_PA
B3.6_PB1
B3.6_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2


In [41]:
# Do the same with Andersen 

# dfs_andersen = create_dataframes(complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/")
dfs_andersen = create_dataframes(complete_files)

A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Cats/Datasets/Andersen/
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Cats/Datasets/Andersen/
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Cats/Datasets/Andersen/
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Cats/Datasets/Andersen/
B3.13_HA
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Cats/Datasets/Andersen/
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Cats/Datasets/Andersen/
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Cats/Datasets/Andersen/
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Cats/Datasets/Andersen/
B3.13_MP
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Cats/Datasets/Andersen/
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Cats/Datasets/Andersen/
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Cats/Datasets/Anderse

In [42]:
# os.chdir(downloads)
# dfs_gisaid["B3.13_HA"].to_csv

In [43]:
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    print(key)

print(dfs_andersen)

A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.2_HA
B3.2_MP
B3.2_NA
B3.2_NP
B3.2_NS
B3.2_PA
B3.2_PB1
B3.2_PB2
B3.5_HA
B3.5_MP
B3.5_NA
B3.5_NP
B3.5_NS
B3.5_PA
B3.5_PB1
B3.5_PB2
B3.6_HA
B3.6_MP
B3.6_NA
B3.6_NP
B3.6_NS
B3.6_PA
B3.6_PB1
B3.6_PB2
B3.7_HA
B3.7_MP
B3.7_NA
B3.7_NP
B3.7_NS
B3.7_PA
B3.7_PB1
B3.7_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
defaultdict(<class 'list'>, {'A3_HA': [   isolate_partial                                        full_header  \
0       004431-001  >SRR33125022|A/CROW/USA/25-004431-001/2025|H5N...   
1       005882-009  >SRR33125049|A/MALLARD/USA/25-005882-009/2025|...   
2       006076-002  >SRR33125051|A/BALD EAGLE/USA/25-006076-002/20...   
3       005882-006  >SRR33125053|A/MALLARD/USA/25-005882-006/2025|...   
4       005882-003  >SRR33125055|A/MALLARD/USA/25-005882-003/2025|...   
5       006076-001  >SRR33125062|A/BALD EAGLE/USA/25-006076-001/20...

In [44]:
print(len(list(dfs_gisaid.keys())))
print(len(list(dfs_andersen.keys())))

32
56


In [37]:
# Merge dataframes and drop duplicates

full_dfs = defaultdict(list)
# same = []
# andersen = set()
# gisaid = set()

for i, andersen_key in enumerate(dfs_andersen.keys()):
    if len(dfs_andersen[andersen_key]) > 0:
        for j, gisaid_key in enumerate(dfs_gisaid.keys()):
            if andersen_key == gisaid_key:
                # same.append(gisaid_key)
        # gisaid_key = list(dfs_gisaid.keys())[i]
        # gisaid2_key = list(dfs_gisaid2.keys())[i]

                andersen_df = dfs_andersen[andersen_key][0]
                print(len(andersen_df))
                # print(andersen_df)
                gisaid_df = dfs_gisaid[gisaid_key][0]
                print(len(gisaid_df))
                # gisaid2_df = dfs_gisaid2[gisaid2_key][0]

                print(pd.concat([gisaid_df, andersen_df]).drop_duplicates())

                full_df = pd.concat([andersen_df, gisaid_df], ignore_index=True)
                print("len full df:", len(full_df))
                test = len(full_df.drop_duplicates(subset="isolate_partial"))

                dedup_df = full_df.drop_duplicates(subset="isolate_partial", keep="last")

                # print((full_df.loc[full_df.duplicated(subset="isolate_partial")]))
                # print((full_df.loc[full_df.duplicated(subset="isolate_partial")]))
                # print(full_df)
                
                print("Keeping nothing: ", test)
                
                print("len deduplicated:", len(dedup_df))
                full_dfs[andersen_key].append(dedup_df)
            # else:
                # gisaid.add(gisaid_key)
                # andersen.add(andersen_key)
    
    # break 


# print(full_dfs)
# print(len(full_dfs))
# print(319*8)
# print(len(same))
# print(len(andersen))
# print(len(gisaid))

18
22
   isolate_partial                                        full_header  \
0        IZ24_0787  >EPI_ISL_19871019|A/northern_pintail/USA/IZ24_...   
1        IZ24_0636  >EPI_ISL_19871011|A/northern_pintail/USA/IZ24_...   
2        IZ24_0570  >EPI_ISL_19871010|A/northern_pintail/USA/IZ24_...   
3        IZ24_0474  >EPI_ISL_19871007|A/northern_pintail/USA/IZ24_...   
4        IZ23_0849  >EPI_ISL_19871006|A/northern_pintail/USA/IZ23_...   
5        IZ23_0847  >EPI_ISL_19871005|A/northern_pintail/USA/IZ23_...   
6       014764-001  >EPI_ISL_19882428|A/hawk/USA/014764-001/2025|H...   
7       005028-001  >EPI_ISL_19870521|A/american_crow/USA/005028-0...   
8       005028-002  >EPI_ISL_19870514|A/barred_owl/USA/005028-002/...   
9       010464-001  >EPI_ISL_19870554|A/common_raven/USA/010464-00...   
10      010511-001  >EPI_ISL_19870553|A/red-tailed_hawk/USA/010511...   
11      010511-002  >EPI_ISL_19870552|A/red-tailed_hawk/USA/010511...   
12      011667-003  >EPI_ISL_19851174|A/ameri

In [47]:
# If none in one database, only use the other and drop duplicates

full_dfs = defaultdict(list)
for key in dfs_andersen.keys():
    print(key)
# for key in ["D1.3"]:
    dataframes = dfs_andersen[key]
    for i, df in enumerate(dataframes):
        print(i)
        try:
            full_df = df.merge(dfs_gisaid[key][i], how="outer")
            # print(full_df)
            full_df = full_df.drop_duplicates(subset=["isolate_partial"])
            full_dfs[key].append(full_df)
        except:
            print("Failed to merge dataframes in ", key)
            full_dfs[key].append(dataframes[i])

A3_HA
0
A3_MP
0
A3_NA
0
A3_NP
0
A3_NS
0
A3_PA
0
A3_PB1
0
A3_PB2
0
B3.13_HA
0
B3.13_MP
0
B3.13_NA
0
B3.13_NP
0
B3.13_NS
0
B3.13_PA
0
B3.13_PB1
0
B3.13_PB2
0
B3.2_HA
0
Failed to merge dataframes in  B3.2_HA
B3.2_MP
0
Failed to merge dataframes in  B3.2_MP
B3.2_NA
0
Failed to merge dataframes in  B3.2_NA
B3.2_NP
0
Failed to merge dataframes in  B3.2_NP
B3.2_NS
0
Failed to merge dataframes in  B3.2_NS
B3.2_PA
0
Failed to merge dataframes in  B3.2_PA
B3.2_PB1
0
Failed to merge dataframes in  B3.2_PB1
B3.2_PB2
0
Failed to merge dataframes in  B3.2_PB2
B3.5_HA
0
Failed to merge dataframes in  B3.5_HA
B3.5_MP
0
Failed to merge dataframes in  B3.5_MP
B3.5_NA
0
Failed to merge dataframes in  B3.5_NA
B3.5_NP
0
Failed to merge dataframes in  B3.5_NP
B3.5_NS
0
Failed to merge dataframes in  B3.5_NS
B3.5_PA
0
Failed to merge dataframes in  B3.5_PA
B3.5_PB1
0
Failed to merge dataframes in  B3.5_PB1
B3.5_PB2
0
Failed to merge dataframes in  B3.5_PB2
B3.6_HA
0
B3.6_MP
0
B3.6_NA
0
B3.6_NP
0
B3.6_NS
0
B3

## Create FASTA files combining Andersen and GISAID

In [48]:
# Create FASTA files per segment

combined_files = downloads + "Cats/Datasets/GISAID_Andersen/" # B3_13_D1_1/" + date_range + "_B3_13_D1_1/"

os.chdir(combined_files)
for pair in full_dfs.keys():
    print(pair)
    output_path = combined_files + pair + "_combined_" + update_date + ".fasta" 

    output_file = open(output_path, "w")
    for item in full_dfs[pair]:
        # for item in item:
        # item = fasta_files[pair]
        for index, row in item.iterrows():
            name = item.loc[index, "full_header"]
            sequence = item.loc[index, "sequence"]
            # print(name)
        # First is header, second is sequence
        # print(value)
            output_file.write(name)
            output_file.write(sequence)
    output_file.close()

A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.2_HA
B3.2_MP
B3.2_NA
B3.2_NP
B3.2_NS
B3.2_PA
B3.2_PB1
B3.2_PB2
B3.5_HA
B3.5_MP
B3.5_NA
B3.5_NP
B3.5_NS
B3.5_PA
B3.5_PB1
B3.5_PB2
B3.6_HA
B3.6_MP
B3.6_NA
B3.6_NP
B3.6_NS
B3.6_PA
B3.6_PB1
B3.6_PB2
B3.7_HA
B3.7_MP
B3.7_NA
B3.7_NP
B3.7_NS
B3.7_PA
B3.7_PB1
B3.7_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
